# Groundwater flow (GWF) model

This notebook loads a pre-built MODFLOW 6 model of a synthetic valley, takes a look at its inputs, runs the groundwater flow (GWF) simulation, and plots the simulated heads. The companion notebook `flopy-intro-gwt-b.ipynb` reuses these flow results to run a groundwater transport (GWT) simulation.

> **Model reference.** The synthetic valley model used in these notebooks is described in Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D., and Brakenhoff, D., 2024, FloPy workflows for creating structured and unstructured MODFLOW models: Groundwater, v. 62, no. 1, p. 124–139, https://doi.org/10.1111/gwat.13327.

## What this notebook covers

Open a model somebody else built, find out what is in it, run it, and read the
heads it produces.

By the end of this notebook you will be able to:

- load an existing MODFLOW 6 simulation with `flopy.mf6.MFSimulation.load()`,
- find a package on the model and read its arrays,
- map hydraulic conductivity, layer elevations, and the water applied at the
  land surface on an unstructured grid,
- add what a transport model needs from a flow run before running it, and
- run the model and plot heads in plan view and in cross section.

The heads and cell-by-cell flows this run saves are what
[`flopy-intro-gwt-b`](flopy-intro-gwt-b.ipynb) reads to drive transport.

## The big picture: how a MODFLOW 6 flow model fits together

Before looking at the inputs, here is the mental model used throughout this notebook and its companion. The whole synthetic-valley example is built on these few ideas, so they are worth a minute now.

**MODFLOW 6 and FloPy**

MODFLOW 6 is the groundwater modeling program: you give it a set of input files and it computes groundwater levels and flows. [FloPy](https://github.com/modflowpy/flopy) is the Python package used here to build, run, and read the output of those models without editing the input files by hand. Every `flopy.mf6.*` object below corresponds to one piece of a MODFLOW 6 model.

**A simulation is a set of nested objects**

- a **simulation** (`sim`) is the top-level container; it holds the time discretization (`TDIS`), the solver (`IMS`), and one or more models;
- a **model** — here a groundwater-flow model (`GWF`) — represents one physical system on one grid;
- each model is made of **packages**, and each package supplies one ingredient: the grid (`DISV`), aquifer properties (`NPF`), storage (`STO`), and the boundaries that add or remove water. Here those are rain soaking through the unsaturated zone to the water table, less evapotranspiration (`UZF`); the stream (`SFR`); the lake (`LAK`); seepage drains that take groundwater where the water table reaches land surface (`DRN`); a water mover (`MVR`) that sends runoff and drain water to the lake or the stream; two pumping wells screened across layers 4 and 5 (multi-aquifer wells, `MAW`); and a well kept idle for a later pumping scenario (`WEL`). Output control (`OC`) decides what gets saved.

**The grid and the layers**

This model uses a **vertex grid** (`DISV`): each layer is divided into the same `ncpl` cells, and the aquifer is stacked into `nlay` layers, so a full 3-D array of model values has shape `(nlay, ncpl)`. Layers 1 and 2 are the upper aquifer and layers 4 and 5 the lower aquifer. Layer 3 is the clay where there is clay; elsewhere its cells have no thickness and pass water straight between layers 2 and 4. Those pass-through cells, and a few layer 1 cells under the lake that also have no thickness, are drawn in light blue on the maps.

**Time is organized in stress periods**

The simulation is divided into **stress periods** — blocks of time over which the stresses are held constant. Here the **first period is a steady-state spin-up** (16,071 days, about 44 years, with the wells off) that establishes a starting condition, and the 20 one-year periods that follow are transient, with the wells pumping. MODFLOW saves the heads it computes at the end of selected time steps; the time slider in the figures below steps through those saved times.

**Units**

This model works in **meters** and **days**: heads and elevations are in meters, hydraulic conductivity and infiltration in meters per day. The figures show meters; set the environment variable `MF6_TRAINING_UNITS` to `feet` before starting Jupyter to see them in feet. The model input stays in meters either way.

Keep this picture — *simulation → model → packages*, a layered grid, stress periods, and meters/days — in mind as you go. The notebook first plots the inputs (properties, geometry, stresses), then runs the model, then maps the simulated heads.

## Imports and helper functions

Import FloPy and the plotting libraries used throughout the notebook. The `time_slider_view` helper (from `mf6_notebook_helpers`) draws a single interactive figure that updates as you move a time slider; it renders each frame to a PNG so exactly one figure is shown in both VS Code and JupyterLab.

In [ ]:
%matplotlib inline
import datetime
import pathlib as pl

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from mf6_notebook_helpers import (
    display_ticks,
    gwf_output_available,
    label_map_axes,
    length_name,
    time_slider_view,
    to_display,
    unit,
)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

## Load the existing model

Read the existing simulation from the `data` directory and point it at a fresh `models/` workspace so the original input files are left untouched. Get the flow model and a few quantities (layer and cell counts, stress-period lengths, and the center of the lake) that the figures below rely on.

In [ ]:
# load the simple model setup

# exercise: set the model name and the workspace to load it from
name = "sv"
ws = pl.Path("../data/synthetic-valley/synthetic-valley-vg-advanced-annual")
new_ws = pl.Path(f"models/{ws.name}")  # end exercise

In [ ]:
# exercise: load the simulation from the data directory and point it at its own workspace
sim = flopy.mf6.MFSimulation.load(
    sim_name=name,
    sim_ws=ws,
    write_headers=False,
)
sim.set_sim_path(new_ws)  # end exercise

In [ ]:
# exercise: get the flow model, and read its grid size and stress periods from it
gwf = sim.get_model(name)
nlay, ncpl = gwf.disv.nlay.array, gwf.disv.ncpl.array
shape3d = (nlay, ncpl)
# end exercise

nper = sim.tdis.nper.array
perlen = sim.tdis.perioddata.array["perlen"]

# x-coordinate of the center of the lake (LAK) package; the model cross
# sections (A-A') are taken along this north-south line so they cut through
# the middle of the lake instead of the center of the model domain
lak_cellids = gwf.lak.connectiondata.array["cellid"]
lak_icpl = np.array(sorted({c[-1] for c in lak_cellids}))
lake_center_x = float(gwf.modelgrid.xcellcenters[lak_icpl].mean())

# map location of the two pumping wells and the layers their screens are in
maw_cellids = gwf.maw.connectiondata.array["cellid"]
maw_icpl = sorted({c[-1] for c in maw_cellids})
maw_layers = sorted({c[0] for c in maw_cellids})
well_x = gwf.modelgrid.xcellcenters[maw_icpl]
well_y = gwf.modelgrid.ycellcenters[maw_icpl]

## Inspect the model inputs

Before running anything, plot the main inputs to get a feel for the model: aquifer properties, geometry, and the stresses applied over time.

### Horizontal hydraulic conductivity

Hydraulic conductivity (K) controls how easily water moves through each layer. Higher K (warmer colors) means more permeable material. The stream is drawn in cyan and the two pumping wells, screened in layers 4 and 5, in red.

In [ ]:
# shared settings for the gwf map figures
gwf_figsize = (17.15 / 2.541, 1.4 * 0.8333 * 17.15 / 2.541)
gwf_mosaic = [[0, 1, 2], [3, 4, "."]]

idomain = gwf.disv.idomain.array

# horizontal hydraulic conductivity by layer, in display units
k11 = to_display(np.ma.masked_where(idomain < 1, gwf.npf.k.array), "rate")
kpos = k11.compressed()
norm_k = mpl.colors.LogNorm(vmin=kpos[kpos > 0].min(), vmax=kpos.max())

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        gwf_mosaic,
        figsize=gwf_figsize,
        layout="constrained",
    )
    fig.suptitle(f"Horizontal hydraulic conductivity ({unit('rate')})")

    for k in range(nlay):
        ax = axd[k]
        ax.set_xlim(gwf.modelgrid.extent[:2])
        ax.set_ylim(gwf.modelgrid.extent[2:])
        ax.set_title(f"Layer {k + 1}")
        mm = flopy.plot.PlotMapView(
            model=gwf,
            ax=ax,
            extent=gwf.modelgrid.extent,
            layer=k,
        )
        mm.plot_ibound(color_vpt="blue", alpha=0.25)
        pa = mm.plot_array(k11, norm=norm_k, edgecolor="black", lw=0.25)
        mm.plot_bc(package=gwf.sfr, color="cyan")
        if k in maw_layers:
            ax.plot(well_x, well_y, "o", mfc="red", mec="black", ms=4)
        label_map_axes(ax, xlabel=False, ylabel=False)
        display_ticks(ax.xaxis, nbins=3)

    fig.colorbar(
        pa,
        ax=[axd[k] for k in range(nlay)],
        shrink=0.6,
        label=f"K ({unit('rate')})",
    )

### Cell bottom elevation

The bottom elevation of every cell, layer by layer. This defines the vertical geometry (and thickness) of the model.

In [ ]:
# cell bottom elevation by layer
botm = to_display(np.ma.masked_where(idomain < 1, gwf.disv.botm.array))

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        gwf_mosaic,
        figsize=gwf_figsize,
        layout="constrained",
    )
    fig.suptitle(f"Cell bottom elevation ({unit()})")

    for k in range(nlay):
        ax = axd[k]
        ax.set_xlim(gwf.modelgrid.extent[:2])
        ax.set_ylim(gwf.modelgrid.extent[2:])
        ax.set_title(f"Layer {k + 1}")
        mm = flopy.plot.PlotMapView(
            model=gwf,
            ax=ax,
            extent=gwf.modelgrid.extent,
            layer=k,
        )
        mm.plot_ibound(color_vpt="blue", alpha=0.25)
        pa = mm.plot_array(
            botm,
            vmin=botm.min(),
            vmax=botm.max(),
            cmap="terrain",
            edgecolor="black",
            lw=0.25,
        )
        mm.plot_bc(package=gwf.sfr, color="cyan")
        label_map_axes(ax, xlabel=False, ylabel=False)
        display_ticks(ax.xaxis, nbins=3)

    fig.colorbar(
        pa,
        ax=[axd[k] for k in range(nlay)],
        shrink=0.6,
        label=f"Elevation ({unit()})",
    )

### Model top and cross-section location

The land-surface elevation across the model; under the lake it is the lake bottom. The lake's outline is drawn in blue. The dashed A–A' line marks the north–south cross section (through the center of the lake) used in the section views below.

In [ ]:
# single figure of the model top
extent = gwf.modelgrid.extent
xext = extent[1] - extent[0]
yext = extent[3] - extent[2]
w = 17.15 / 2.541 / 2
single_figsize = (w, w * yext / xext)

top = to_display(np.ma.masked_where(idomain[0] < 1, gwf.disv.top.array))

# cross-section location through the center of the lake (matches the
# cross-section figures)
xc = lake_center_x

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=single_figsize, layout="constrained")
    ax.set_xlim(extent[:2])
    ax.set_ylim(extent[2:])
    ax.set_title(f"Model top ({unit()})")
    mm = flopy.plot.PlotMapView(
        model=gwf,
        ax=ax,
        extent=extent,
        layer=0,
    )
    mm.plot_ibound(color_vpt="blue", alpha=0.25)
    pa = mm.plot_array(top, cmap="terrain", edgecolor="black", lw=0.25)
    mm.plot_bc(package=gwf.sfr, color="cyan")
    # outline of the cells under the lake
    lake_cells = np.zeros(ncpl)
    lake_cells[lak_icpl] = 1.0
    mm.contour_array(lake_cells, levels=[0.5], colors="blue", linewidths=1.0)
    label_map_axes(ax, xlabel=False, ylabel=False)
    display_ticks(ax.xaxis, nbins=4)

    # show the cross-section line
    ax.plot([xc, xc], extent[2:], color="red", lw=1.5, ls="--")
    ax.text(
        xc, extent[3], "A", color="red", fontweight="bold", ha="center", va="bottom"
    )
    ax.text(xc, extent[2], "A'", color="red", fontweight="bold", ha="center", va="top")

    fig.colorbar(pa, ax=ax, shrink=0.8, label=f"Elevation ({unit()})")

### Hydraulic conductivity along section A–A'

The same hydraulic conductivity, now shown in cross section along A–A' so you can see how the layers stack vertically.

In [ ]:
# cross-section along section A-A' (center of the lake) showing horizontal K
extent = gwf.modelgrid.extent
xc = lake_center_x
# order the line A (north/top) -> A' (south/bottom) to match the map labels
line = {"line": [(xc, extent[3]), (xc, extent[2])]}

w = 17.15 / 2.541
with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(w, 0.5 * w), layout="constrained")
    ax.set_title("Horizontal hydraulic conductivity along section A-A'")
    xs = flopy.plot.PlotCrossSection(
        model=gwf,
        ax=ax,
        line=line,
    )
    pa = xs.plot_array(to_display(gwf.npf.k.array, "rate"), norm=norm_k)
    xs.plot_grid(lw=0.3, color="0.5")
    xs.plot_ibound()
    label_map_axes(ax, xlabel=False, ylabel=False)
    ax.set_xlabel(f"Distance along section, in {length_name()}")
    ax.set_ylabel(f"Elevation, in {length_name()}")

    # label the section endpoints to match the model-top figure
    ax.text(
        0.0,
        1.01,
        "A",
        color="red",
        fontweight="bold",
        ha="left",
        va="bottom",
        transform=ax.transAxes,
    )
    ax.text(
        1.0,
        1.01,
        "A'",
        color="red",
        fontweight="bold",
        ha="right",
        va="bottom",
        transform=ax.transAxes,
    )

    fig.colorbar(pa, ax=ax, shrink=0.8, label=f"K ({unit('rate')})")

### Water applied at the land surface

Rain on the land cells soaks into the unsaturated zone at the infiltration rate the unsaturated-zone flow (UZF) package reads from a time-series file, one value per year. On its way down, and from the water table where it is within 3.0 m (10 ft) of land surface, evapotranspiration takes water up to the potential rate, also plotted. What is left reaches the water table as recharge. Rain on the lake goes to the lake package instead. The plot shows the 20 transient years; the steady-state period holds one rate of each constant.

In [ ]:
# infiltration and potential evapotranspiration on the land cells, from the
# unsaturated-zone time series; each value holds until the next time
series = gwf.uzf.ts.timeseries.get_data()
finf, pet = series["ts_array_0"], series["ts_array_1"]  # uzf_finf, uzf_pet (m/d)

# times are days since START_DATE_TIME = 1962-01-01; keep the transient years
start_date = datetime.datetime(1962, 1, 1)
transient = series["ts_time"] >= perlen[0]
dates = [
    start_date + datetime.timedelta(days=float(t)) for t in series["ts_time"][transient]
]

# a rate in m/d times 365.25 d is the depth applied in a year
finf_year = to_display(finf[transient] * 365.25)
pet_year = to_display(pet[transient] * 365.25)

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(
        figsize=(17.15 / 2.541, 0.45 * 17.15 / 2.541),
        layout="constrained",
    )
    ax.plot(
        dates,
        finf_year,
        drawstyle="steps-post",
        color="blue",
        lw=1.0,
        label="Infiltration",
    )
    ax.fill_between(dates, finf_year, step="post", color="blue", alpha=0.2)
    ax.plot(
        dates,
        pet_year,
        drawstyle="steps-post",
        color="green",
        lw=1.0,
        ls="--",
        label="Potential evapotranspiration",
    )
    ax.set_xlim(dates[0], dates[-1])
    ax.set_ylim(bottom=0.0)
    ax.legend(loc="upper left")
    flopy.plot.styles.heading(
        ax=ax,
        heading="Water applied to the land during the transient years",
    )
    flopy.plot.styles.xlabel(
        ax=ax,
        label="Date",
    )
    flopy.plot.styles.ylabel(
        ax=ax,
        label=f"Rate, in {unit()} per year",
    )

## Run the flow model

Before running, add the two things the transport notebook reads from this run. Then write the input files and run the flow simulation, but only if the output is missing or incomplete. If the head and budget files already contain every saved time step, the run is skipped so re-running the notebook stays fast; a full run takes a minute or two.

### Get the flow output ready for transport

A transport model does not solve for flow; it reads the flow model's budget file. Two things it needs are not saved by this model as built:

- **a concentration for the lake water.** An auxiliary variable is an extra number attached to a boundary that MODFLOW does not use for flow but writes to the budget file with every flow of that boundary. Give the lake an auxiliary variable named `CONCENTRATION` with a value of 100, and the transport notebook can treat every flow from the lake into the aquifer as water at that concentration. With 100, a simulated concentration reads as the percent of the water that came from the lake.
- **how full each cell is.** Transport needs the saturated fraction of each cell to know how much water it holds, so the flow model has to save it.

In [ ]:
# label the lake water: one value per lake, read by the transport model
lakes = gwf.lak.packagedata.array
gwf.lak.auxiliary = ["concentration"]
gwf.lak.packagedata = [
    (lake["ifno"], lake["strt"], lake["nlakeconn"], 100.0, lake["boundname"])
    for lake in lakes
]

# save the saturated fraction of each cell for the transport model
gwf.npf.save_saturation = True

In [ ]:
# only run the gwf model if its output is missing or incomplete, so re-running
# the notebook stays fast. gwf_output_available (in mf6_notebook_helpers) checks that
# the head and budget files exist and contain every time step the OC saves.
# exercise: write and run the flow model, but only when its output is missing or incomplete
if gwf_output_available(gwf, sim, new_ws):
    print("gwf output is already complete; skipping sim.run_simulation()")
else:
    sim.write_simulation()
    success, buff = sim.run_simulation()
    assert success, "MODFLOW 6 did not terminate normally"

## View the simulated heads

Plot the simulated groundwater levels (heads). Use the time slider to step through the simulation.

### Head maps through time

Simulated head in each layer at the selected time. Contours show equal-head lines, and the A–A' line marks the cross section shown below.

In [ ]:
# simulated heads through time (six-panel map; one frame per output time)
head_times = gwf.output.head().get_times()

extent = gwf.modelgrid.extent
xc = lake_center_x  # cross-section through the center of the lake (A-A')


def display_head(totim):
    """Simulated head at a time in display units, with inactive cells masked."""
    head = gwf.output.head().get_data(totim=totim)
    return to_display(np.ma.masked_greater(head, 1e29))


# one color scale and set of contours for every time, from the range of the
# heads over the whole simulation
all_heads = to_display(np.ma.masked_greater(gwf.output.head().get_alldata(), 1e29))
head_vmin, head_vmax = np.floor(all_heads.min()), np.ceil(all_heads.max())
head_levels = mpl.ticker.MaxNLocator(nbins=10).tick_values(head_vmin, head_vmax)
head_ticks = head_levels

head_mosaic = [[0, 1, 2], [3, 4, "cbar"]]


def plot_head(time_index):
    totim = head_times[time_index]
    head = display_head(totim)

    with flopy.plot.styles.USGSMap():
        fig, axd = plt.subplot_mosaic(
            head_mosaic,
            figsize=gwf_figsize,
            layout="constrained",
        )
        fig.suptitle(
            f"Simulated head ({unit()}), time = {totim:g} days (index {time_index})"
        )

        for k in range(nlay):
            ax = axd[k]
            ax.set_xlim(extent[:2])
            ax.set_ylim(extent[2:])
            ax.set_title(f"Layer {k + 1}")
            mm = flopy.plot.PlotMapView(
                model=gwf,
                ax=ax,
                extent=extent,
                layer=k,
            )
            mm.plot_ibound(color_vpt="blue", alpha=0.25)
            pa = mm.plot_array(
                head,
                vmin=head_vmin,
                vmax=head_vmax,
                edgecolor="black",
                lw=0.25,
            )
            mm.plot_bc(package=gwf.sfr, color="cyan")
            if k in maw_layers:
                ax.plot(well_x, well_y, "o", mfc="red", mec="black", ms=4)
            label_map_axes(ax, xlabel=False, ylabel=False)
            display_ticks(ax.xaxis, nbins=3)

            cs = mm.contour_array(
                head,
                levels=head_levels,
                colors="black",
                linewidths=0.5,
            )
            ax.clabel(
                cs,
                inline=True,
                fmt="%g",
                fontsize=8,
                inline_spacing=0.5,
            )

            # show the cross-section location on the Layer 1 panel
            if k == 0:
                ax.plot([xc, xc], extent[2:], color="red", lw=1.5, ls="--")
                ax.text(
                    xc,
                    extent[3],
                    "A",
                    color="red",
                    fontweight="bold",
                    ha="center",
                    va="bottom",
                )
                ax.text(
                    xc,
                    extent[2],
                    "A'",
                    color="red",
                    fontweight="bold",
                    ha="center",
                    va="top",
                )

        # colorbar in the empty mosaic panel
        axd["cbar"].axis("off")
        cax = inset_axes(axd["cbar"], width="25%", height="85%", loc="center")
        cb = fig.colorbar(pa, cax=cax, ticks=head_ticks)
        cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}"))
        cb.set_label(f"Head ({unit()})")

    return fig


time_slider_view(plot_head, len(head_times))

**What to look for.** Drag the slider from the first saved time to the last. At the end of the steady-state period (index 0) heads run from 0.28 to 7.56 m (0.93 to 24.79 ft), highest in the uplands east of the lake and lowest at the southern edge of the valley. Heads slope from the uplands toward the lake, the stream (cyan), and the south; along the stream the contours bend upstream, the sign of groundwater seeping into it. Once the wells (red) start pumping, heads around the southwestern well, which pumps 7,589 m³/d (268,000 ft³/d), drop into a local low in layers 4 and 5. By the last saved time heads run from -1.20 to 4.97 m (-3.94 to 16.31 ft), lowest at that well. The cell that moves most, in the eastern uplands, swings 2.84 m (9.30 ft) over the simulation as wet and dry years raise and lower the water table.

### Head along section A–A' through time

Simulated head in cross section along A–A', so you can watch the water table rise and fall through time.

In [ ]:
# simulated heads along section A-A' (center of the lake) through time
line = {"line": [(xc, extent[3]), (xc, extent[2])]}  # A (top) -> A' (bottom)
xs_w = 17.15 / 2.541


def plot_head_xs(time_index):
    totim = head_times[time_index]
    # heads in meters set the water table; the colors are in display units
    head = gwf.output.head().get_data(totim=totim)
    head_shown = display_head(totim)

    with flopy.plot.styles.USGSMap():
        fig, ax = plt.subplots(figsize=(xs_w, 0.5 * xs_w), layout="constrained")
        ax.set_title(
            f"Simulated head along A-A', time = {totim:g} days (index {time_index})"
        )
        xs = flopy.plot.PlotCrossSection(
            model=gwf,
            ax=ax,
            line=line,
        )
        pa = xs.plot_array(
            head_shown,
            head=head,
            vmin=head_vmin,
            vmax=head_vmax,
        )
        xs.plot_grid(lw=0.3, color="0.5")
        xs.plot_ibound()

        cs = xs.contour_array(
            head_shown,
            head=head,
            levels=head_levels,
            colors="black",
            linewidths=0.5,
        )
        ax.clabel(cs, inline=True, fmt="%g", fontsize=8, inline_spacing=0.5)

        label_map_axes(ax, xlabel=False, ylabel=False)
        ax.set_xlabel(f"Distance along section, in {length_name()}")
        ax.set_ylabel(f"Elevation, in {length_name()}")

        # label the section endpoints to match the map figure
        ax.text(
            0.0,
            1.01,
            "A",
            color="red",
            fontweight="bold",
            ha="left",
            va="bottom",
            transform=ax.transAxes,
        )
        ax.text(
            1.0,
            1.01,
            "A'",
            color="red",
            fontweight="bold",
            ha="right",
            va="bottom",
            transform=ax.transAxes,
        )

        cb = fig.colorbar(pa, ax=ax, shrink=0.8, ticks=head_ticks)
        cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}"))
        cb.set_label(f"Head ({unit()})")

    return fig


time_slider_view(plot_head_xs, len(head_times))

**What to look for.** In cross section the colored fill is head and the black lines are equal-head contours, drawn only in the saturated part of each cell. A–A' runs north to south through the lake, which covers roughly the first 2,400 m (7,900 ft) of the section. Under the lake the head in the aquifer is a little below the lake level, so lake water leaks down into the aquifer; south of the ridge the heads fall toward the southern edge, and the nearly vertical contours in the lower aquifer show mostly horizontal flow. Stepping through time shows the heads under the lake following the lake level down, from 7.11 m (23.34 ft) at the end of the steady-state period to 4.89 m (16.05 ft) at the end of the simulation.

**Recap.** You loaded a pre-built MODFLOW 6 flow model with FloPy, inspected its properties, geometry, and stresses, gave the lake water a concentration and saved the cell saturation for transport, ran the steady-state-plus-transient flow simulation, and mapped the resulting heads in plan view and cross section. The saved heads and cell-by-cell flows from this run are exactly what the companion notebook `flopy-intro-gwt-b.ipynb` reads back in to drive a solute-transport simulation on the same grid.